# Provision Deduplication

This notebook is to test whether `bundesrecht.normalise()` can turn real-world legal citation strings into stable provision identities.

The source is `webapp/sorted_law_references.txt`: `550,657` unique citation strings extracted from real text. This experiment does not invent synthetic variants from clean provisions. Instead, it mines real observed strings into high-confidence variant sets using an independent conservative parser.

The core question:

**Given real surface forms that look different, can the library identify when they point to the same provision?**

Why this matters:

- citation counts should count legal provisions, not spelling variants
- retrieval labels should point to stable targets, not surface strings
- graph edges should connect legal units, not surface citation text
- failures reveal concrete normalizer gaps on real observed data


In [1]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".."], check=True)
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from bundesrecht import normalise

## Design

This is a large-scale version of the same two checks:

1. **Canonical target accuracy**: does `normalise(surface_form)` produce the independently assigned canonical target?
2. **Dedup clustering**: do real surface forms in the same mined group collapse together, while different groups stay separate?

The canonical targets are not produced by `normalise()`. They are produced by a small independent parser that accepts only conservative single-provision shapes: section marker, section number, optional Absatz/Satz/Nr/Buchstabe components, and a law abbreviation at the end.

Rows with connectors, ranges, multi-section markers, obvious malformed prefixes, or leftover text are rejected before evaluation. This keeps the mined labels high-confidence and makes the results interpretable.


In [2]:
from collections import Counter, defaultdict
from pathlib import Path
import re

FF_EXPANSION = 3

REFERENCE_FILE_CANDIDATES = [
    Path("../../webapp/sorted_law_references.txt"),
    Path("../webapp/sorted_law_references.txt"),
    Path("webapp/sorted_law_references.txt"),
]

reference_file = next((p for p in REFERENCE_FILE_CANDIDATES if p.exists()), None)
if reference_file is None:
    raise FileNotFoundError("Could not find webapp/sorted_law_references.txt")

surface_forms = [
    line.strip()
    for line in reference_file.read_text(errors="replace").splitlines()
    if line.strip()
]

print(f"Reference file : {reference_file}")
print(f"Surface forms  : {len(surface_forms):,}")
print(f"Unique surface forms    : {len(set(surface_forms)):,}")

Reference file : ../../webapp/sorted_law_references.txt
Surface forms  : 550,657
Unique surface forms    : 550,657


## Independent Group Miner

The miner creates canonical targets without calling `normalise()`.
It deliberately handles only a narrow grammar:

- `§ 433 BGB`
- `§433 BGB`
- `§ 433 Abs. 1 BGB`
- `§ 433 Absatz 1 BGB`
- `§ 433 I BGB`
- `§ 91 Abs. 1 S. 1 ZPO`
- `§ 2 Abs. 1 Nr. 1 UrhG`
- `§ 2 Abs. 1 Ziffer 1 UrhG`
- `§ 81 Abs. 1 Nr. 1 lit. a BGB`

The miner also mirrors stable library-contract choices that are already public in examples and tests, such as Roman suffixes in law names becoming Arabic
suffixes (`SGB VIII` to `SGB 8`) and embedded Nummer letters becoming Buchstabe references (`Nr. 2b` to `Nr. 2 Buchst. b`).


In [3]:
ROMAN = {
    "I": "1",
    "II": "2",
    "III": "3",
    "IV": "4",
    "V": "5",
    "VI": "6",
    "VII": "7",
    "VIII": "8",
    "IX": "9",
    "X": "10",
    "XI": "11",
    "XII": "12",
}

LAW_ALLOWED = re.compile(
    r"^[A-ZÄÖÜ][A-Za-zÄÖÜäöüß0-9]*(?:-[A-Za-zÄÖÜäöüß0-9]+)*(?:\s+[IVX]{1,8})?$"
)
CONNECTOR_OR_RANGE = re.compile(
    r"\b(und|oder|bis|i\.?\s*v\.?\s*m\.?|ivm|ff?\.?)\b", re.I
)
MULTI_OR_NOISE = re.compile(r"[,;]|^§§|\b(den|die|das|der)\b", re.I)
SECTION_SHAPE = re.compile(
    r"^§\s*([0-9]+[a-z]?)\s+(.*?)\s+([A-ZÄÖÜ][A-Za-zÄÖÜäöüß0-9-]*(?:\s+[IVX]{1,8})?)$"
)

LEVEL_PATTERNS = [
    ("Abs", re.compile(r"(?:^|\s)(?:Abs\.?|Absatz|Ab\.?)\s*(\d+[a-z]?)\b", re.I)),
    ("Satz", re.compile(r"(?:^|\s)(?:S\.?|Satz)\s*(\d+)\b", re.I)),
    (
        "Nr",
        re.compile(r"(?:^|\s)(?:Nr\.?|Nummer|Ziffer|Ziff\.?)\s*(\d+[a-z]?)\b", re.I),
    ),
    (
        "Buchst",
        re.compile(r"(?:^|\s)(?:Buchst\.?|Buchstabe|lit\.?)\s*([a-z]{1,2})\b", re.I),
    ),
]


def canonical_law(law: str) -> str:
    """Apply the library's public Roman-law-suffix convention."""
    parts = law.split()
    if parts and parts[-1] in ROMAN:
        parts[-1] = ROMAN[parts[-1]]
    return " ".join(parts)


def independent_canonical_target(raw: str) -> tuple[str | None, str]:
    """Return an independent canonical target, or a rejection reason."""
    text = " ".join(raw.strip().split())

    if not text.startswith("§") or text.startswith("§§"):
        return None, "not_single_section"
    if MULTI_OR_NOISE.search(text):
        return None, "multi_or_noise"
    if CONNECTOR_OR_RANGE.search(text):
        return None, "connector_or_range"

    match = SECTION_SHAPE.match(text)
    if not match:
        return None, "shape"

    section, middle, law_raw = match.groups()
    if not LAW_ALLOWED.match(law_raw):
        return None, "law"

    law = canonical_law(law_raw)

    if re.search(r"(^|\s)\d+[a-z]?\s+\d+", middle):
        return None, "middle_numeric_noise"

    if re.search(r"\b[IVXLCDM]{1,8}\b", middle):
        tokens = middle.split()
        if (
            len(tokens) in (1, 2)
            and tokens[0] in ROMAN
            and (len(tokens) == 1 or tokens[1].isdigit())
        ):
            parts = [f"§ {section}", f"Abs. {ROMAN[tokens[0]]}"]
            if len(tokens) == 2:
                parts.append(f"Satz {tokens[1]}")
            parts.append(law)
            return " ".join(parts), "ok"
        return None, "roman_noise"

    subrefs = []
    leftover = middle
    for level, pattern in LEVEL_PATTERNS:
        match_level = pattern.search(middle)
        if not match_level:
            continue

        value = (
            match_level.group(1).lower() if level == "Buchst" else match_level.group(1)
        )
        if level == "Nr":
            embedded = re.fullmatch(r"(\d+)([a-z])", value, re.I)
            if embedded:
                subrefs.append(("Nr", embedded.group(1)))
                subrefs.append(("Buchst", embedded.group(2).lower()))
            else:
                subrefs.append((level, value))
        else:
            subrefs.append((level, value))

        leftover = pattern.sub(" ", leftover, count=1)

    if re.sub(r"[.()\s]", "", leftover):
        return None, "leftover"

    parts = [f"§ {section}"]
    for level, value in subrefs:
        if level == "Abs":
            parts.append(f"Abs. {value}")
        elif level == "Satz":
            parts.append(f"Satz {value}")
        elif level == "Nr":
            parts.append(f"Nr. {value}")
        elif level == "Buchst":
            parts.append(f"Buchst. {value}")
    parts.append(law)
    return " ".join(parts), "ok"

In [4]:
mined_groups = defaultdict(set)
reject_reasons = Counter()

for raw in surface_forms:
    canonical_target, reason = independent_canonical_target(raw)
    if canonical_target is None:
        reject_reasons[reason] += 1
    else:
        mined_groups[canonical_target].add(raw)

variant_sets = {
    canonical_target: sorted(variants)
    for canonical_target, variants in mined_groups.items()
    if len(variants) >= 2
}

accepted_surface_forms = sum(len(variants) for variants in mined_groups.values())
surface_forms_in_variant_sets = sum(len(variants) for variants in variant_sets.values())

print(
    f"Accepted surface forms       : {accepted_surface_forms:,} ({accepted_surface_forms / len(surface_forms):.2%})"
)
print(f"Accepted canonical targets   : {len(mined_groups):,}")
print(f"Variant sets                : {len(variant_sets):,}")
print(f"Surface forms in variant sets: {surface_forms_in_variant_sets:,}")
print()
print("Rejected surface forms by reason")
print("-" * 64)
for reason, count in reject_reasons.most_common():
    print(f"{reason:<24} {count:>10,} ({count / len(surface_forms):.2%})")

Accepted surface forms       : 248,716 (45.17%)
Accepted canonical targets   : 199,249
Variant sets                : 35,010
Surface forms in variant sets: 84,477

Rejected surface forms by reason
----------------------------------------------------------------
not_single_section          122,746 (22.29%)
connector_or_range           49,579 (9.00%)
shape                        43,542 (7.91%)
multi_or_noise               39,716 (7.21%)
leftover                     38,586 (7.01%)
middle_numeric_noise          4,402 (0.80%)
roman_noise                   3,370 (0.61%)


## Mined Variant Examples

These variant sets are mined from real observed surface forms. The canonical target is produced by the independent parser, the surface forms are the strings found in the file.

In [5]:
print(f"{'SURFACE FORMS':>13}  CANONICAL TARGET")
print("-" * 90)
for canonical_target, variants in sorted(
    variant_sets.items(), key=lambda item: len(item[1]), reverse=True
)[:12]:
    print(f"{len(variants):>13}\t{canonical_target}")
    for raw in variants[:5]:
        print(f"\t\t{raw}")
    print()

SURFACE FORMS  CANONICAL TARGET
------------------------------------------------------------------------------------------
           16	§ 113 Abs. 1 Satz 1 VwGO
		§ 113 Abs 1 Satz 1 VwGO
		§ 113 Abs. 1 S. 1 VwGO
		§ 113 Abs. 1 S. 1. VwGO
		§ 113 Abs. 1 S.1 VwGO
		§ 113 Abs. 1 Satz 1 VwGO

           15	§ 540 Abs. 1 Satz 1 Nr. 1 ZPO
		§ 540 Abs. 1 Nr. 1 S. 1 ZPO
		§ 540 Abs. 1 S. 1 Nr. 1 ZPO
		§ 540 Abs. 1 S. 1. Nr. 1 ZPO
		§ 540 Abs. 1 S.1 Nr. 1 ZPO
		§ 540 Abs. 1 S.1 Nr.1 ZPO

           14	§ 86b Abs. 1 Satz 1 Nr. 2 SGG
		§ 86b Abs 1 Satz 1 Nr 2 SGG
		§ 86b Abs 1 Satz 1 Nr. 2 SGG
		§ 86b Abs. 1 S. 1 Nr. 2 SGG
		§ 86b Abs. 1 S.1 Nr. 2 SGG
		§ 86b Abs. 1 S.1 Nr.2 SGG

           13	§ 7 Abs. 1 Satz 2 Nr. 2 SGB 2
		§ 7 Abs 1 Satz 2 Nr 2 SGB II
		§ 7 Abs 1 Satz 2 Nr. 2 SGB II
		§ 7 Abs. 1 S. 2 Nr. 2 SGB II
		§ 7 Abs. 1 S. 2 Nr.2 SGB II
		§ 7 Abs. 1 Satz 2 Nr. 2 SGB II

           13	§ 91 Abs. 1 Satz 1 ZPO
		§ 91 Abs 1 S. 1 ZPO
		§ 91 Abs 1 Satz 1 ZPO
		§ 91 Abs. 1 S. 1 ZPO
		§ 91 Abs. 1 S

## Layer 1: Canonical Target Accuracy

For each mined surface form, compare the library output against the independently mined canonical target.

A row is correct only if:

```python
frozenset(normalise(raw, ff_expansion=3)) == frozenset({canonical_target})
```

This is the direct answer to: can the library recover the intended provision identity from real-world surface variants?

In [6]:
def depth_bucket(canonical_target: str) -> str:
    if "Buchst." in canonical_target:
        return "buchstabe"
    if "Nr." in canonical_target:
        return "nummer"
    if "Satz" in canonical_target:
        return "satz"
    if "Abs." in canonical_target:
        return "absatz"
    return "section"


accuracy_rows = []
for canonical_target, variants in variant_sets.items():
    canonical_target_set = frozenset({canonical_target})
    for raw in variants:
        actual = frozenset(normalise(raw, ff_expansion=FF_EXPANSION))
        accuracy_rows.append(
            {
                "canonical_target": canonical_target,
                "raw": raw,
                "actual": actual,
                "correct": actual == canonical_target_set,
                "bucket": depth_bucket(canonical_target),
            }
        )

n_total = len(accuracy_rows)
n_correct = sum(row["correct"] for row in accuracy_rows)

print(
    f"Canonical target accuracy : {n_correct:,}/{n_total:,} = {n_correct / n_total:.2%}"
)
print()
print(f"{'BUCKET':<12} {'CORRECT':>10} {'TOTAL':>10} {'ACCURACY':>10}")
print("-" * 48)
for bucket in ["absatz", "satz", "nummer", "buchstabe", "section"]:
    rows = [row for row in accuracy_rows if row["bucket"] == bucket]
    if not rows:
        continue
    correct = sum(row["correct"] for row in rows)
    print(f"{bucket:<12} {correct:>10,} {len(rows):>10,} {correct / len(rows):>9.2%}")

Canonical target accuracy : 83,625/84,477 = 98.99%

BUCKET          CORRECT      TOTAL   ACCURACY
------------------------------------------------
absatz           25,953     25,963    99.96%
satz             43,931     43,963    99.93%
nummer           12,843     13,576    94.60%
buchstabe           898        975    92.10%


## Canonical Target Accuracy Failures

Failures are useful: they show where the library does not currently preserve the full independently mined provision identity.

The main failure pattern is that some references containing `Satz` and `Nr.` are split into two targets instead of one nested target. Embedded Nummer letters and Buchstabe/lit forms also expose remaining edge cases.

In [7]:
bad_rows = [row for row in accuracy_rows if not row["correct"]]
print(f"Mismatches: {len(bad_rows):,}")
print()
for row in bad_rows[:20]:
    print(f"SURFACE FORM : {row['raw']}")
    print(f"CANONICAL TARGET : {row['canonical_target']}")
    print(f"ACTUAL   : {sorted(row['actual'])}")
    print()

Mismatches: 852

SURFACE FORM : § 1 Abs. 2 Nr. 1 Satz 1 WoGG
CANONICAL TARGET : § 1 Abs. 2 Satz 1 Nr. 1 WoGG
ACTUAL   : ['§ 1 Abs. 2 Nr. 1 WoGG', '§ 1 Abs. 2 Satz 1 WoGG']

SURFACE FORM : § 1 Abs. 1 Nr. 1 S. 1 UStG
CANONICAL TARGET : § 1 Abs. 1 Satz 1 Nr. 1 UStG
ACTUAL   : ['§ 1 Abs. 1 Nr. 1 UStG', '§ 1 Abs. 1 Satz 1 UStG']

SURFACE FORM : § 1 Abs. 1 Nr. 1 Satz 1 UStG
CANONICAL TARGET : § 1 Abs. 1 Satz 1 Nr. 1 UStG
ACTUAL   : ['§ 1 Abs. 1 Nr. 1 UStG', '§ 1 Abs. 1 Satz 1 UStG']

SURFACE FORM : § 1 Abs.1 Nr. 1 Satz 1 UStG
CANONICAL TARGET : § 1 Abs. 1 Satz 1 Nr. 1 UStG
ACTUAL   : ['§ 1 Abs. 1 Nr. 1 UStG', '§ 1 Abs. 1 Satz 1 UStG']

SURFACE FORM : § 1 Abs.1 Nr.1 Satz 1 UStG
CANONICAL TARGET : § 1 Abs. 1 Satz 1 Nr. 1 UStG
ACTUAL   : ['§ 1 Abs. 1 Nr. 1 UStG', '§ 1 Abs. 1 Satz 1 UStG']

SURFACE FORM : § 1 Abs. 1 Nr. 1 S. 2 DrittelbG
CANONICAL TARGET : § 1 Abs. 1 Satz 2 Nr. 1 DrittelbG
ACTUAL   : ['§ 1 Abs. 1 Nr. 1 DrittelbG', '§ 1 Abs. 1 Satz 2 DrittelbG']

SURFACE FORM : § 1 Abs. 1 Nr. 1 Sa

## Layer 2: Dedup Clustering

Layer 1 checks whether each surface form maps to the right target. Layer 2 checks whether the deduplication behavior is good at group level.

Because the mined set has `84,477` rows, pairwise metrics are computed with combinatorics rather than by enumerating billions of pairs.

In [8]:
def raw_key(raw: str) -> str:
    """Baseline key: only lowercase and collapse whitespace."""
    return " ".join(raw.lower().split())


def canonical_key(raw: str):
    targets = normalise(raw, ff_expansion=FF_EXPANSION)
    if not targets:
        return None
    return tuple(sorted(targets))


def choose2(n: int) -> int:
    return n * (n - 1) // 2


def clustering_metrics(items, predicted_key):
    canonical_target_counts = Counter()
    predicted_counts = Counter()
    joint_counts = Counter()
    n = 0

    for canonical_target, raw in items:
        predicted = predicted_key(raw)
        if predicted is None:
            continue
        n += 1
        canonical_target_counts[canonical_target] += 1
        predicted_counts[predicted] += 1
        joint_counts[(canonical_target, predicted)] += 1

    true_positive = sum(choose2(count) for count in joint_counts.values())
    predicted_same = sum(choose2(count) for count in predicted_counts.values())
    canonical_target_same = sum(
        choose2(count) for count in canonical_target_counts.values()
    )
    total_pairs = choose2(n)

    false_positive = predicted_same - true_positive
    false_negative = canonical_target_same - true_positive
    true_negative = total_pairs - true_positive - false_positive - false_negative

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive
        else 1.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative
        else 1.0
    )
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "n": n,
        "pairs": total_pairs,
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
        "tn": true_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


items = [
    (canonical_target, raw)
    for canonical_target, variants in variant_sets.items()
    for raw in variants
]
raw_metrics = clustering_metrics(items, raw_key)
canonical_metrics = clustering_metrics(items, canonical_key)

print(f"Evaluated surface forms : {canonical_metrics['n']:,}")
print(f"Pair count     : {canonical_metrics['pairs']:,}")
print()
print(
    f"{'SYSTEM':<22} {'PRECISION':>10} {'RECALL':>10} {'F1':>10} {'TP':>12} {'FP':>8} {'FN':>8}"
)
print("-" * 92)
for label, metrics in [
    ("surface-form baseline", raw_metrics),
    ("canonical", canonical_metrics),
]:
    print(
        f"{label:<22} "
        f"{metrics['precision']:>10.3f} "
        f"{metrics['recall']:>10.3f} "
        f"{metrics['f1']:>10.3f} "
        f"{metrics['tp']:>12,} "
        f"{metrics['fp']:>8,} "
        f"{metrics['fn']:>8,}"
    )

Evaluated surface forms : 84,477
Pair count     : 3,568,139,526

SYSTEM                  PRECISION     RECALL         F1           TP       FP       FN
--------------------------------------------------------------------------------------------
surface-form baseline       0.530      0.002      0.004          132      117   72,734
canonical                   1.000      0.990      0.995       72,158        0      708


The experiment mines `35,010` real-world variant sets from `sorted_law_references.txt`, covering `84,477` observed surface forms with high-confidence independent labels.

Results:

- canonical target accuracy is `98.99%`
- surface-form deduplication has almost no recall (`0.002`)
- canonical deduplication has high recall (`0.990`) and perfect precision on the mined labels
